# 🏛️ Huấn luyện Mô hình PhoBERT-CRF trên Google Colab

Notebook này tự động mount **Google Drive**, kiểm tra thư mục dự án `VPhoBertTagger`, nạp bộ dữ liệu **PAP_NER** và thực hiện huấn luyện mô hình **PhoBERT + CRF** với siêu tham số chuẩn từ bài báo (*PLOS ONE* - Dev F1: **97.82%**).

### 📌 Cấu hình huấn luyện tối ưu:
- **Backbone:** `vinai/phobert-base` (Contextual Embedding 768-dim)
- **Classifier Head:** `Linear(768 -> 11)` + `CRF` (Viterbi Decoding)
- **PhoBERT Learning Rate:** `2e-5` | **CRF Learning Rate:** `1e-3`
- **Scheduler:** Linear decay + 10% Warmup | **Batch size:** `32` | **Epochs:** `20` (Early stop: `3`)

## Bước 1: Kiểm tra GPU Google Colab
Đảm bảo Colab đang bật GPU (khuyên dùng T4, V100 hoặc A100).

In [1]:
!nvidia-smi

## Bước 2: Mount Google Drive & Tự động Định vị Thư mục `VPhoBertTagger`

In [3]:
# 1. Mount Google Drive
import os
import sys
from pathlib import Path

try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        drive.mount('/content/drive')
    print('✅ Đã kết nối Google Drive thành công!')
except Exception as e:
    print(f'ℹ️ Ghi chú khi mount Drive: {e}')

# 2. Tìm kiếm thư mục VPhoBertTagger trên Google Drive
candidate_paths = [
    '/content/drive/MyDrive/VPhoBertTagger',
    '/content/drive/MyDrive/PHAPLUAT/VPhoBertTagger',
    '/content/drive/MyDrive/phapluat/VPhoBertTagger'
]

project_dir = None
for p in candidate_paths:
    if os.path.exists(p) and os.path.exists(os.path.join(p, 'main.py')):
        project_dir = p
        break

if project_dir is None:
    print('🔍 Đang tự động quét tìm thư mục VPhoBertTagger trong Drive...')
    search_root = '/content/drive/MyDrive'
    if os.path.exists(search_root):
        for root, dirs, files in os.walk(search_root):
            if 'VPhoBertTagger' in dirs and os.path.exists(os.path.join(root, 'VPhoBertTagger', 'main.py')):
                project_dir = os.path.join(root, 'VPhoBertTagger')
                break
            if root.count(os.sep) - search_root.count(os.sep) >= 3:
                dirs.clear()

if project_dir and os.path.exists(project_dir):
    os.chdir(project_dir)
    try:
        get_ipython().run_line_magic('cd', project_dir)
    except Exception:
        pass
    print(f'\n✅ ĐÃ ĐỊNH VỊ VÀ CHUYỂN VÀO THƯ MỤC DỰ ÁN:\n👉 {os.getcwd()}')
else:
    raise FileNotFoundError('❌ Không tìm thấy thư mục VPhoBertTagger trên Google Drive! Vui lòng kiểm tra lại vị trí đặt thư mục trên Drive.')

# Thêm vào PYTHONPATH
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())

# Liệt kê các file trong thư mục dự án
!ls -la


✅ Đã kết nối Google Drive thành công!
🔍 Đang tự động quét tìm thư mục VPhoBertTagger trong Drive...


FileNotFoundError: ❌ Không tìm thấy thư mục VPhoBertTagger trên Google Drive! Vui lòng kiểm tra lại vị trí đặt thư mục trên Drive.

In [ ]:
# 📂 Liệt kê các thư mục có trong Google Drive (MyDrive)
import os

drive_root = '/content/drive/MyDrive'

if not os.path.exists(drive_root):
    print('❌ Chưa kết nối Google Drive! Vui lòng chạy cell mount drive ở trên trước.')
else:
    print('====================================================')
    print(f'📂 DANH SÁCH CÁC THƯ MỤC TRONG DRIVE: {drive_root}')
    print('====================================================\n')
    try:
        top_items = sorted([f for f in os.listdir(drive_root) if os.path.isdir(os.path.join(drive_root, f))])
        for idx, folder in enumerate(top_items, 1):
            full_p = os.path.join(drive_root, folder)
            tag = ' ⭐ [THƯ MỤC DỰ ÁN]' if 'VPhoBertTagger' in folder else ''
            print(f'{idx:2d}. 📁 {folder}{tag}')
            # Quét thư mục con cấp 1
            try:
                subs = sorted([sf for sf in os.listdir(full_p) if os.path.isdir(os.path.join(full_p, sf))])
                for sf in subs[:8]:
                    sub_tag = ' ⭐ [THƯ MỤC DỰ ÁN]' if 'VPhoBertTagger' in sf else ''
                    print(f'       └── 📁 {sf}{sub_tag}')
            except Exception:
                pass
    except Exception as err:
        print(f'Lỗi khi đọc thư mục Drive: {err}')


## Bước 3: Cài đặt Thư viện Phụ thuộc (Dependencies)

In [ ]:
!pip install -q transformers>=4.28.0 pytorch-crf>=0.7.2 prettytable tensorboardX vncorenlp
!pip install -q -r requirements.txt
print("\n✅ Cài đặt thư viện hoàn tất!")

## Bước 4: Kiểm tra Dữ liệu PAP_NER (`Dataset/`)
Xác nhận sự tồn tại của `train_data.txt`, `dev_data.txt`, `test_data.txt`.

In [ ]:
dataset_dir = Path("Dataset")
assert (dataset_dir / "train_data.txt").exists(), "Thiếu file train_data.txt trong Dataset/!"
assert (dataset_dir / "dev_data.txt").exists(), "Thiếu file dev_data.txt trong Dataset/!"
assert (dataset_dir / "test_data.txt").exists(), "Thiếu file test_data.txt trong Dataset/!"

print("✅ Cả 3 file dữ liệu PAP_NER đã sẵn sàng!")
!ls -lh Dataset/*.txt

print("\n--- 5 DÒNG MẪU ĐẦU TIÊN TỪ DEV_DATA.TXT ---")
with open('Dataset/dev_data.txt', 'r', encoding='utf-8') as f:
    for _ in range(5):
        print(f.readline().strip())

## Bước 5: Mở TensorBoard Theo dõi Quá trình Huấn luyện

In [ ]:
%load_ext tensorboard
%tensorboard --logdir runs

## Bước 6: Thực hiện Huấn luyện Mô hình PhoBERT-CRF (Fine-Tuning)
Toàn bộ siêu tham số chuẩn từ bài báo (*PLOS ONE*):
- Model: `vinai/phobert-base` + `crf`
- PhoBERT LR: `2e-5` | CRF LR: `1e-3`
- Batch Size: `32` | Epochs: `20` | Early stop: `3` epochs

In [ ]:
!python main.py train \
    --task pap_ner \
    --data_dir ./Dataset \
    --model_name_or_path vinai/phobert-base \
    --model_arch crf \
    --output_dir outputs \
    --max_seq_length 256 \
    --train_batch_size 32 \
    --eval_batch_size 32 \
    --learning_rate 2e-5 \
    --classifier_learning_rate 1e-3 \
    --epochs 20 \
    --early_stop 3 \
    --overwrite_data \
    --run_test

## Bước 7: Đánh giá Độc lập trên Tập Test (CoNLL Strict Evaluation)

In [ ]:
!python main.py test --data_dir ./Dataset --model_path outputs/best_model.pt

## Bước 8: Tải VnCoreNLP & Dự đoán Thực thể trên Văn bản Mới (Inference)

In [ ]:
# Tải VnCoreNLP jar nếu chưa có
if not os.path.exists("vncorenlp/VnCoreNLP-1.1.1.jar"):
    !mkdir -p vncorenlp/models/wordsegmenter
    !wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/VnCoreNLP-1.1.1.jar -O vncorenlp/VnCoreNLP-1.1.1.jar
    !wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/vi-vocab -O vncorenlp/models/wordsegmenter/vi-vocab
    !wget -q https://raw.githubusercontent.com/vncorenlp/VnCoreNLP/master/models/wordsegmenter/wordsegmenter.rdr -O vncorenlp/models/wordsegmenter/wordsegmenter.rdr
    print("✅ Tải VnCoreNLP thành công!")
else:
    print("✅ VnCoreNLP đã có sẵn!")

In [ ]:
from vphoberttagger.predictor import ViTagger

# Nạp mô hình vừa huấn luyện
tagger = ViTagger(model_path="outputs/best_model.pt")

# Thử nghiệm câu văn bản pháp luật / hành chính thực tế:
sample_text = "Ủy ban nhân dân tỉnh Thái Bình ban hành Nghị định 123/2020/NĐ-CP ngày 15 tháng 10 năm 2020 quy định về hồ sơ đăng ký doanh nghiệp tư nhân với lệ phí 5 triệu đồng."

entities = tagger(sample_text)
print("\n--- KẾT QUẢ TRÍCH XUẤT THỰC THỂ (LEGAL ENTITIES) ---")
for entity, label in entities:
    print(f"{entity:40} -> [{label}]")